# 🏠 Sistem Rekomendasi Kos — Preprocessing & Forward Chaining

**Mata Kuliah:** Sistem Berbasis Pengetahuan  
**Politeknik Negeri Malang — Jurusan Teknologi Informasi**  

---

Notebook ini mencakup:
1. **Preprocessing Data** dari dataset Mamikos Jabodetabek
2. **Pemodelan Forward Chaining** untuk rekomendasi kos

> **Dataset:** Mamikos_Jabodetabek_Data.xlsx — 2.485 data kos di wilayah Jabodetabek

## 📦 Install & Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')

## 📂 Load Dataset

> Upload file `Mamikos_Jabodetabek_Data.xlsx` ke Google Colab terlebih dahulu

In [ ]:
from google.colab import files

# Upload file Excel
uploaded = files.upload()

In [ ]:
df_raw = pd.read_excel('Mamikos_Jabodetabek_Data.xlsx')
print(f'Shape dataset: {df_raw.shape}')
print(f'Kolom: {df_raw.columns.tolist()}')
df_raw.head()

## 🔧 PREPROCESSING DATA

### Step 1: Eksplorasi Awal

In [ ]:
print('=== INFO DATASET ===')
print(df_raw.info())
print()
print('=== MISSING VALUES ===')
print(df_raw.isnull().sum())
print()
print('=== DISTRIBUSI TIPE KOS ===')
print(df_raw['tipe_kos'].value_counts())

### Step 2: Cleaning — Konversi Kolom Price

In [ ]:
def parse_price(price_str):
    """Konversi string harga 'Rp1.500.000' ke integer 1500000"""
    if pd.isna(price_str) or price_str == 'Not found':
        return np.nan
    cleaned = re.sub(r'[Rp.\s]', '', str(price_str))
    try:
        return int(cleaned)
    except:
        return np.nan

df = df_raw.copy()
df['price_numeric'] = df['price'].apply(parse_price)
print('Sample konversi harga:')
print(df[['price', 'price_numeric']].head(10))

### Step 3: Cleaning — Konversi Rating

In [ ]:
def parse_rating(r):
    if pd.isna(r) or str(r).strip() == 'Not found':
        return np.nan
    try:
        return float(r)
    except:
        return np.nan

df['rating_numeric'] = df['rating'].apply(parse_rating)

# Isi missing rating dengan median
median_rating = df['rating_numeric'].median()
df['rating_numeric'] = df['rating_numeric'].fillna(median_rating)
print(f'Median rating digunakan untuk mengisi NaN: {median_rating}')
print(df['rating_numeric'].describe())

### Step 4: Cleaning — Konversi Room Size

In [ ]:
def parse_room_size(size_str):
    """Ekstrak luas kamar dalam m² dari string seperti '3 x 4 meter'"""
    if pd.isna(size_str):
        return np.nan
    match = re.search(r'([\d.]+)\s*[xX]\s*([\d.]+)', str(size_str))
    if match:
        return float(match.group(1)) * float(match.group(2))
    return np.nan

df['room_size_m2'] = df['room_size'].apply(parse_room_size)
median_size = df['room_size_m2'].median()
df['room_size_m2'] = df['room_size_m2'].fillna(median_size)
print(f'Median ukuran kamar: {median_size} m²')
print(df['room_size_m2'].describe())

### Step 5: Feature Engineering — Ekstraksi Fasilitas

In [ ]:
def has_facility(facilities_str, keyword):
    """Cek apakah fasilitas tertentu ada dalam string fasilitas"""
    if pd.isna(facilities_str):
        return 0
    return 1 if keyword.lower() in str(facilities_str).lower() else 0

facility_keywords = {
    'has_ac': 'AC',
    'has_wifi': 'WiFi',
    'has_private_bathroom': 'K. Mandi Dalam',
    'has_parking': 'Parkir',
    'has_cctv': 'CCTV',
    'has_laundry': 'Laundry',
    'has_kitchen': 'Dapur',
    'has_wardrobe': 'Lemari Baju',
    'has_bed': 'Kasur'
}

for col, keyword in facility_keywords.items():
    df[col] = df['all_facilities_bs'].apply(lambda x: has_facility(x, keyword))

# Hitung total fasilitas
facility_cols = list(facility_keywords.keys())
df['total_facilities'] = df[facility_cols].sum(axis=1)

print('Distribusi fasilitas:')
print(df[facility_cols].sum().sort_values(ascending=False))

### Step 6: Konversi Kolom Numerik Lainnya & Drop Rows Invalid

In [ ]:
def parse_numeric(val):
    if pd.isna(val) or str(val).strip() == 'Not found':
        return 0
    try:
        return int(float(str(val).replace(',', '')))
    except:
        return 0

df['rating_count_num'] = df['rating_count'].apply(parse_numeric)
df['transaction_count_num'] = df['transaction_count'].apply(parse_numeric)

# Flag listrik
df['electricity_included'] = df['is_electricity_included'].apply(
    lambda x: 1 if 'termasuk' in str(x).lower() else 0
)

# Drop baris yang price_numeric NaN
df = df.dropna(subset=['price_numeric'])
df = df[df['price_numeric'] > 0]
print(f'Shape setelah cleaning: {df.shape}')

### Step 7: Kategorisasi & Label Encoding

In [ ]:
# Kategori harga
def kategorisasi_harga(harga):
    if harga < 1_000_000:
        return 'murah'
    elif harga < 2_000_000:
        return 'sedang'
    elif harga < 3_500_000:
        return 'mahal'
    else:
        return 'sangat_mahal'

# Kategori rating
def kategorisasi_rating(rating):
    if rating >= 4.5:
        return 'sangat_baik'
    elif rating >= 4.0:
        return 'baik'
    elif rating >= 3.0:
        return 'cukup'
    else:
        return 'kurang'

# Kategori ukuran kamar
def kategorisasi_ukuran(m2):
    if m2 >= 16:
        return 'luas'
    elif m2 >= 9:
        return 'sedang'
    else:
        return 'sempit'

# Kategori fasilitas
def kategorisasi_fasilitas(total):
    if total >= 7:
        return 'lengkap'
    elif total >= 4:
        return 'standar'
    else:
        return 'minim'

df['kategori_harga'] = df['price_numeric'].apply(kategorisasi_harga)
df['kategori_rating'] = df['rating_numeric'].apply(kategorisasi_rating)
df['kategori_ukuran'] = df['room_size_m2'].apply(kategorisasi_ukuran)
df['kategori_fasilitas'] = df['total_facilities'].apply(kategorisasi_fasilitas)

print('=== Distribusi Kategori Harga ===')
print(df['kategori_harga'].value_counts())
print()
print('=== Distribusi Kategori Rating ===')
print(df['kategori_rating'].value_counts())

### Step 8: Buat Dataset Bersih Final

In [ ]:
kolom_final = [
    'room_name', 'region', 'tipe_kos',
    'price_numeric', 'rating_numeric', 'room_size_m2',
    'total_facilities', 'electricity_included',
    'has_ac', 'has_wifi', 'has_private_bathroom',
    'has_parking', 'has_cctv', 'has_laundry',
    'has_kitchen', 'has_wardrobe', 'has_bed',
    'rating_count_num', 'transaction_count_num',
    'kategori_harga', 'kategori_rating',
    'kategori_ukuran', 'kategori_fasilitas'
]

df_clean = df[kolom_final].reset_index(drop=True)
print(f'Dataset bersih: {df_clean.shape}')
df_clean.head()

In [ ]:
# Simpan dataset bersih
df_clean.to_csv('mamikos_clean.csv', index=False)
print('Dataset bersih disimpan sebagai mamikos_clean.csv')
print(f'Total data: {len(df_clean)} baris')
print()
print('=== Statistik Deskriptif ===')
df_clean[['price_numeric', 'rating_numeric', 'room_size_m2', 'total_facilities']].describe()

---
## 🔁 FORWARD CHAINING

### Konsep
Forward Chaining (penalaran maju) bekerja dari **fakta → aturan → kesimpulan**.
Sistem membaca preferensi pengguna sebagai fakta, lalu mencocokkan dengan aturan (rule) untuk menghasilkan rekomendasi kos.

**Alur:**
```
Fakta (Input Preferensi) → Evaluasi Rule 1..N → Kesimpulan (Rekomendasi)
```

### Definisi Knowledge Base (Rule Base)

In [ ]:
# ============================================================
# KNOWLEDGE BASE — Rule untuk Forward Chaining Rekomendasi Kos
# ============================================================

# Rule Base: list of (kondisi_dict, label_rekomendasi, skor_bobot)
# Kondisi menggunakan key yang sesuai dengan kolom di dataframe

RULE_BASE = [
    # Rule 1: Mahasiswa hemat — butuh kos murah dengan fasilitas minimal
    {
        'kondisi': {'kategori_harga': 'murah', 'kategori_rating': ['baik', 'sangat_baik']},
        'label': 'Cocok untuk Mahasiswa Budget',
        'bobot': 3
    },
    # Rule 2: Profesional nyaman — butuh AC, WiFi, kamar mandi dalam
    {
        'kondisi': {'has_ac': 1, 'has_wifi': 1, 'has_private_bathroom': 1},
        'label': 'Cocok untuk Profesional',
        'bobot': 4
    },
    # Rule 3: Premium — mahal dengan rating sangat baik
    {
        'kondisi': {'kategori_harga': ['mahal', 'sangat_mahal'], 'kategori_rating': 'sangat_baik', 'kategori_fasilitas': 'lengkap'},
        'label': 'Kos Premium',
        'bobot': 5
    },
    # Rule 4: Aman — ada CCTV dan listrik included
    {
        'kondisi': {'has_cctv': 1, 'electricity_included': 1},
        'label': 'Kos Aman & Nyaman',
        'bobot': 3
    },
    # Rule 5: Lengkap fasilitas
    {
        'kondisi': {'kategori_fasilitas': 'lengkap', 'kategori_rating': ['baik', 'sangat_baik']},
        'label': 'Kos Fasilitas Lengkap',
        'bobot': 4
    },
    # Rule 6: Mahasiswi (putri) dengan budget sedang
    {
        'kondisi': {'tipe_kos': 'Kos Putri', 'kategori_harga': 'sedang', 'has_wifi': 1},
        'label': 'Cocok untuk Mahasiswi',
        'bobot': 4
    },
    # Rule 7: Kos keluarga / campur dengan kamar luas
    {
        'kondisi': {'tipe_kos': 'Kos Campur', 'kategori_ukuran': 'luas'},
        'label': 'Kos Keluarga / Pasangan',
        'bobot': 3
    },
    # Rule 8: Value for money — sedang harga, rating baik
    {
        'kondisi': {'kategori_harga': 'sedang', 'kategori_rating': 'sangat_baik'},
        'label': 'Value for Money',
        'bobot': 4
    },
]

print(f'Total rules di Knowledge Base: {len(RULE_BASE)}')
for i, rule in enumerate(RULE_BASE, 1):
    print(f'Rule {i}: {rule["label"]} (bobot={rule["bobot"]})')

### Implementasi Forward Chaining Engine

In [ ]:
def evaluate_rule(row, kondisi):
    """Cek apakah sebuah baris data memenuhi semua kondisi dalam rule."""
    for field, value in kondisi.items():
        if field not in row.index:
            return False
        row_val = row[field]
        if isinstance(value, list):
            if row_val not in value:
                return False
        else:
            if row_val != value:
                return False
    return True


def forward_chaining_engine(df, rule_base):
    """
    Forward Chaining:
    Untuk setiap kos, cek semua rule.
    Akumulasikan skor bobot dari setiap rule yang terpenuhi.
    Label rekomendasi diambil dari rule dengan bobot tertinggi yang terpenuhi.
    """
    results = []
    
    for idx, row in df.iterrows():
        matched_rules = []
        total_score = 0
        
        # === FORWARD: Evaluasi setiap rule dari atas ke bawah ===
        for rule in rule_base:
            if evaluate_rule(row, rule['kondisi']):
                matched_rules.append(rule['label'])
                total_score += rule['bobot']
        
        # Tentukan label akhir
        if matched_rules:
            # Ambil label dari rule dengan bobot tertinggi yang match
            best_rule = max(
                [r for r in rule_base if r['label'] in matched_rules],
                key=lambda x: x['bobot']
            )
            label_akhir = best_rule['label']
        else:
            label_akhir = 'Tidak Direkomendasikan'
        
        results.append({
            'idx': idx,
            'fc_score': total_score,
            'fc_label': label_akhir,
            'fc_rules_matched': len(matched_rules),
            'fc_rules_list': ', '.join(matched_rules) if matched_rules else 'Tidak ada'
        })
    
    return pd.DataFrame(results).set_index('idx')


print('Menjalankan Forward Chaining...')
fc_results = forward_chaining_engine(df_clean, RULE_BASE)
print('Selesai!')
print(f'Total kos diproses: {len(fc_results)}')
print()
print('=== Distribusi Label Rekomendasi ===')
print(fc_results['fc_label'].value_counts())

### Gabungkan Hasil & Tampilkan Rekomendasi

In [ ]:
df_fc = df_clean.copy()
df_fc = df_fc.join(fc_results)

# Filter hanya yang direkomendasikan, urutkan berdasarkan skor
df_recommended = df_fc[df_fc['fc_label'] != 'Tidak Direkomendasikan'].copy()
df_recommended = df_recommended.sort_values('fc_score', ascending=False)

print(f'Total kos direkomendasikan: {len(df_recommended)}')
print(f'Total kos tidak direkomendasikan: {len(df_fc) - len(df_recommended)}')

### Fungsi Rekomendasi Interaktif Berdasarkan Preferensi

In [ ]:
def rekomendasi_forward_chaining(df_fc, preferensi_user, top_n=10):
    """
    Mencari rekomendasi kos berdasarkan preferensi user.
    
    preferensi_user: dict dengan key seperti:
        - budget_max: int (harga maksimum)
        - tipe_kos: str ('Kos Putri', 'Kos Putra', 'Kos Campur')
        - region: str (wilayah)
        - min_rating: float
        - fasilitas: list (e.g. ['has_ac', 'has_wifi'])
    """
    result = df_fc.copy()
    
    # === FAKTA → Filter berdasarkan preferensi (Forward dari kondisi user) ===
    if 'budget_max' in preferensi_user:
        result = result[result['price_numeric'] <= preferensi_user['budget_max']]
    
    if 'tipe_kos' in preferensi_user:
        result = result[result['tipe_kos'] == preferensi_user['tipe_kos']]
    
    if 'region' in preferensi_user:
        result = result[result['region'].str.contains(preferensi_user['region'], case=False, na=False)]
    
    if 'min_rating' in preferensi_user:
        result = result[result['rating_numeric'] >= preferensi_user['min_rating']]
    
    if 'fasilitas' in preferensi_user:
        for fac in preferensi_user['fasilitas']:
            if fac in result.columns:
                result = result[result[fac] == 1]
    
    # Urutkan: skor FC tertinggi, lalu rating
    result = result.sort_values(['fc_score', 'rating_numeric'], ascending=[False, False])
    
    kolom_tampil = ['room_name', 'region', 'tipe_kos', 'price_numeric',
                    'rating_numeric', 'room_size_m2', 'fc_score', 'fc_label', 'fc_rules_list']
    
    return result[kolom_tampil].head(top_n)


# =====================================================
# CONTOH PENGGUNAAN 1: Mahasiswi budget sedang, butuh WiFi & AC
# =====================================================
print('=== CONTOH 1: Kos untuk Mahasiswi Budget Sedang ===')
preferensi_1 = {
    'budget_max': 2_000_000,
    'tipe_kos': 'Kos Putri',
    'min_rating': 4.0,
    'fasilitas': ['has_wifi']
}
hasil_1 = rekomendasi_forward_chaining(df_fc, preferensi_1, top_n=5)
print(hasil_1.to_string())

In [ ]:
# =====================================================
# CONTOH PENGGUNAAN 2: Profesional di Jakarta Selatan
# =====================================================
print('=== CONTOH 2: Kos untuk Profesional di Jakarta Selatan ===')
preferensi_2 = {
    'budget_max': 4_000_000,
    'region': 'Jakarta Selatan',
    'min_rating': 4.5,
    'fasilitas': ['has_ac', 'has_wifi', 'has_private_bathroom']
}
hasil_2 = rekomendasi_forward_chaining(df_fc, preferensi_2, top_n=5)
print(hasil_2.to_string())

In [ ]:
# =====================================================
# CONTOH PENGGUNAAN 3: Kos Budget Murah Bekasi
# =====================================================
print('=== CONTOH 3: Kos Budget Murah di Bekasi ===')
preferensi_3 = {
    'budget_max': 1_000_000,
    'region': 'Bekasi'
}
hasil_3 = rekomendasi_forward_chaining(df_fc, preferensi_3, top_n=5)
print(hasil_3.to_string())

### Visualisasi Distribusi Hasil Forward Chaining

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Distribusi Label
label_counts = df_fc['fc_label'].value_counts()
axes[0].barh(label_counts.index, label_counts.values, color='steelblue')
axes[0].set_title('Distribusi Label Rekomendasi\n(Forward Chaining)', fontweight='bold')
axes[0].set_xlabel('Jumlah Kos')

# Plot 2: Distribusi FC Score
axes[1].hist(df_fc['fc_score'], bins=15, color='coral', edgecolor='black')
axes[1].set_title('Distribusi Skor Forward Chaining', fontweight='bold')
axes[1].set_xlabel('Skor')
axes[1].set_ylabel('Frekuensi')

# Plot 3: Jumlah Rules Matched
rule_counts = df_fc['fc_rules_matched'].value_counts().sort_index()
axes[2].bar(rule_counts.index, rule_counts.values, color='mediumseagreen')
axes[2].set_title('Jumlah Rules yang Terpenuhi per Kos', fontweight='bold')
axes[2].set_xlabel('Jumlah Rules Match')
axes[2].set_ylabel('Jumlah Kos')

plt.tight_layout()
plt.savefig('forward_chaining_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualisasi disimpan!')

### Evaluasi & Ringkasan Forward Chaining

In [ ]:
print('=' * 60)
print('RINGKASAN FORWARD CHAINING')
print('=' * 60)
print(f'Total data kos    : {len(df_fc)}')
print(f'Total rules KB    : {len(RULE_BASE)}')
print(f'Kos direkomendasikan  : {len(df_fc[df_fc["fc_label"] != "Tidak Direkomendasikan"])} ({len(df_fc[df_fc["fc_label"] != "Tidak Direkomendasikan"])/len(df_fc)*100:.1f}%)')
print(f'Kos tidak direkomendasikan: {len(df_fc[df_fc["fc_label"] == "Tidak Direkomendasikan"])} ({len(df_fc[df_fc["fc_label"] == "Tidak Direkomendasikan"])/len(df_fc)*100:.1f}%)')
print()
print('Top 5 Kos dengan Skor FC Tertinggi:')
top5 = df_fc.nlargest(5, 'fc_score')[['room_name', 'region', 'price_numeric', 'rating_numeric', 'fc_score', 'fc_label']]
print(top5.to_string())
print()
print('Kelebihan Forward Chaining:')
print('  - Transparan: dapat melacak rule mana yang aktif')
print('  - Cepat untuk jumlah rule yang kecil')
print('  - Mudah dipahami dan dijelaskan kepada pengguna')
print()
print('Keterbatasan Forward Chaining:')
print('  - Tidak memperhitungkan ketidakpastian')
print('  - Rule harus dibuat manual oleh pakar')
print('  - Skor bersifat kumulatif sederhana, tidak ada normalisasi')